### DataFrame Operations

In [2]:
import spark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StringType, StructField, IntegerType

#### Create Spark Session

In [3]:
spark = SparkSession.builder\
        .appName("DataFrame_Operations")  \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 16:06:38 INFO SparkEnv: Registering MapOutputTracker
26/04/11 16:06:38 INFO SparkEnv: Registering BlockManagerMaster
26/04/11 16:06:38 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/11 16:06:39 INFO SparkEnv: Registering OutputCommitCoordinator


#### Sample Data

In [4]:
data = [
        (1, "Alice", 25),
        (2, "Bob", 30),
        (3, "Charlie", 35)
        ]

#### Define Schema

In [5]:
schema = StructType([
                    StructField("id", IntegerType(), False),
                    StructField("name", StringType(), False),
                    StructField("age", IntegerType(), False),
                    ])

#### Create DataFrame

In [47]:
df = spark.createDataFrame(data, schema)

In [48]:
df.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



In [49]:
df.printSchema()

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- age: integer (nullable = false)



In [35]:
df.columns

['id', 'name', 'age']

In [36]:
df.describe().show()

+-------+---+-------+----+
|summary| id|   name| age|
+-------+---+-------+----+
|  count|  3|      3|   3|
|   mean|2.0|   NULL|30.0|
| stddev|1.0|   NULL| 5.0|
|    min|  1|  Alice|  25|
|    max|  3|Charlie|  35|
+-------+---+-------+----+



In [37]:
df.select('name','age').show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [38]:
df.filter(df.age >= 30).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



In [39]:
df.where(df.name == "Alice").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
+---+-----+---+



In [40]:
df.distinct().show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



#### Sorting and ordering

In [41]:
df.orderBy(df.age).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



In [42]:
df.orderBy(df.age.desc()).show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  3|Charlie| 35|
|  2|    Bob| 30|
|  1|  Alice| 25|
+---+-------+---+



#### Adding and dropping columns

In [50]:
#adding
df = df.withColumn('new_age', df.age+5)

In [51]:
df.show()

+---+-------+---+-------+
| id|   name|age|new_age|
+---+-------+---+-------+
|  1|  Alice| 25|     30|
|  2|    Bob| 30|     35|
|  3|Charlie| 35|     40|
+---+-------+---+-------+



In [52]:
# renaming the above age column name
df = df.withColumnRenamed('age', 'old_age')

In [53]:
df.show()

+---+-------+-------+-------+
| id|   name|old_age|new_age|
+---+-------+-------+-------+
|  1|  Alice|     25|     30|
|  2|    Bob|     30|     35|
|  3|Charlie|     35|     40|
+---+-------+-------+-------+



In [54]:
#dropping
df = df.drop('old_age')

In [55]:
df.show()

+---+-------+-------+
| id|   name|new_age|
+---+-------+-------+
|  1|  Alice|     30|
|  2|    Bob|     35|
|  3|Charlie|     40|
+---+-------+-------+



#### Aggregation and Grouping

In [57]:
## grouping
df.groupBy('name').count().show()

+-------+-----+
|   name|count|
+-------+-----+
|  Alice|    1|
|Charlie|    1|
|    Bob|    1|
+-------+-----+



In [59]:
## agg
df.agg({'new_age': 'avg'}).show()

+------------+
|avg(new_age)|
+------------+
|        35.0|
+------------+



In [60]:
df.agg({'new_age': 'max'}).show()

+------------+
|max(new_age)|
+------------+
|          40|
+------------+



#### Joins

In [66]:
data2 = [(1,'usa'),
         (2,'uk'),
         (3,'India')
        ]

In [67]:
schema2 = StructType([
                    StructField("id", IntegerType(), True),
                    StructField("country", StringType(), True)
                    ])

In [68]:
df2 = spark.createDataFrame(data2, schema2)

In [69]:
df2.show()

+---+-------+
| id|country|
+---+-------+
|  1|    usa|
|  2|     uk|
|  3|  India|
+---+-------+



In [65]:
df.show()

+---+-------+-------+
| id|   name|new_age|
+---+-------+-------+
|  1|  Alice|     30|
|  2|    Bob|     35|
|  3|Charlie|     40|
+---+-------+-------+



In [72]:
final_df = df.join(df2, "id")

In [73]:
final_df.show()

+---+-------+-------+-------+
| id|   name|new_age|country|
+---+-------+-------+-------+
|  1|  Alice|     30|    usa|
|  2|    Bob|     35|     uk|
|  3|Charlie|     40|  India|
+---+-------+-------+-------+



In [74]:
spark.stop()